In [2]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


raw_df = spark.read.option("multiline", "true").json(
    "Files/bronze/weather/raw/weather_test.json"
)

display(raw_df)

StatementMeta(, e696a93b-f1c3-4adc-91ea-38f30ac455af, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ef5e0dbc-2944-4a1b-9947-6d61b55e9f9c)

In [3]:
from pyspark.sql import functions as F

silver_df = (
    raw_df
    .select(
        F.explode(
            F.arrays_zip(
                F.col("hourly.time"),
                F.col("hourly.temperature_2m")
            )
        ).alias("weather")
    )
    .select(
        F.col("weather.time").alias("observation_time"),
        F.col("weather.temperature_2m").alias("temperature_2m")
    )
)

display(silver_df)

StatementMeta(, e696a93b-f1c3-4adc-91ea-38f30ac455af, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c4f0f019-1a40-4ad2-bc0e-29a9aee0295d)

In [4]:
silver_df = (
    silver_df
    .withColumn(
        "observation_time",
        F.to_timestamp("observation_time")
    )
    .withColumn(
        "temperature_2m",
        F.col("temperature_2m").cast("double")
    )
)

StatementMeta(, e696a93b-f1c3-4adc-91ea-38f30ac455af, 6, Finished, Available, Finished, False)

In [5]:
silver_df.printSchema()
display(silver_df)

StatementMeta(, e696a93b-f1c3-4adc-91ea-38f30ac455af, 7, Finished, Available, Finished, False)

root
 |-- observation_time: timestamp (nullable = true)
 |-- temperature_2m: double (nullable = true)



SynapseWidget(Synapse.DataFrame, 56801f0b-c71b-438f-93d2-0421357a8897)

In [6]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_weather_observations")

StatementMeta(, e696a93b-f1c3-4adc-91ea-38f30ac455af, 8, Finished, Available, Finished, False)

In [7]:
display(
    spark.table("silver_weather_observations")
)

StatementMeta(, e696a93b-f1c3-4adc-91ea-38f30ac455af, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 46a5634f-2103-4e37-a3b0-acb7e19b941b)

In [8]:
raw_df = (
    spark.read
    .option("multiline", "true")
    .json("Files/bronze/weather/raw/weather_test.json")
)

display(raw_df)

StatementMeta(, e696a93b-f1c3-4adc-91ea-38f30ac455af, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 81f6d023-792f-4c5a-a27f-868f7d9d9891)

In [9]:
from pyspark.sql import functions as F

silver_df = (
    raw_df
    .select(
        F.explode(
            F.arrays_zip(
                F.col("hourly.time"),
                F.col("hourly.temperature_2m"),
                F.col("hourly.relative_humidity_2m"),
                F.col("hourly.precipitation"),
                F.col("hourly.wind_speed_10m")
            )
        ).alias("weather")
    )
    .select(
        F.col("weather.time").alias("observation_time"),
        F.col("weather.temperature_2m").alias("temperature_2m"),
        F.col("weather.relative_humidity_2m").alias("relative_humidity_2m"),
        F.col("weather.precipitation").alias("precipitation"),
        F.col("weather.wind_speed_10m").alias("wind_speed_10m")
    )
)


StatementMeta(, e696a93b-f1c3-4adc-91ea-38f30ac455af, 11, Finished, Available, Finished, False)

In [10]:
silver_df = (
    silver_df
    .withColumn("observation_time", F.to_timestamp("observation_time"))
    .withColumn("temperature_2m", F.col("temperature_2m").cast("double"))
    .withColumn("relative_humidity_2m", F.col("relative_humidity_2m").cast("integer"))
    .withColumn("precipitation", F.col("precipitation").cast("double"))
    .withColumn("wind_speed_10m", F.col("wind_speed_10m").cast("double"))
)

display(silver_df)

StatementMeta(, e696a93b-f1c3-4adc-91ea-38f30ac455af, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 814d6293-50ee-4604-b474-794155bb7369)

In [11]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_weather_observations")

StatementMeta(, e696a93b-f1c3-4adc-91ea-38f30ac455af, 13, Finished, Available, Finished, False)

In [12]:
from delta.tables import DeltaTable

silver_table = DeltaTable.forName(
    spark,
    "silver_weather_observations"
)

(
    silver_table.alias("target")
    .merge(
        silver_df.alias("source"),
        "target.observation_time = source.observation_time"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("Silver MERGE completed successfully.")

StatementMeta(, e696a93b-f1c3-4adc-91ea-38f30ac455af, 14, Finished, Available, Finished, False)

Silver MERGE completed successfully.


In [13]:
display(
    spark.table("silver_weather_observations")
)

StatementMeta(, e696a93b-f1c3-4adc-91ea-38f30ac455af, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4f697d80-b359-4170-b196-e633497361c7)

In [15]:
silver_check = spark.table("silver_weather_observations")

silver_check.agg(
    F.count("*").alias("row_count"),
    F.countDistinct("observation_time").alias("unique_observation_times"),
    F.min("observation_time").alias("earliest_observation"),
    F.max("observation_time").alias("latest_observation")
).show()

StatementMeta(, e696a93b-f1c3-4adc-91ea-38f30ac455af, 17, Finished, Available, Finished, False)

+---------+------------------------+--------------------+-------------------+
|row_count|unique_observation_times|earliest_observation| latest_observation|
+---------+------------------------+--------------------+-------------------+
|       48|                      48| 2026-08-04 00:00:00|2026-08-05 23:00:00|
+---------+------------------------+--------------------+-------------------+

